# Import Libraries, Dependencies & Datasets

In [ ]:
!pip install scikeras
!pip install keras_tuner


In [ ]:
import numpy as np
import pandas as pd
import itertools
import seaborn as sns
import nltk, re, string
import matplotlib.pyplot as plt

from collections import defaultdict
from string import punctuation
from nltk.corpus import stopwords

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, GlobalMaxPool1D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.regularizers import l2
import keras_tuner as kt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV

from scikeras.wrappers import KerasClassifier

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from keras.models import Model
from keras.layers import Dense, Embedding, Input, LSTM, Bidirectional, GlobalMaxPool1D, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
nltk.download("stopwords");

In [ ]:
# Importiamo i Dati che ci serviranno per addestrare il Modello
data_fake = pd.read_csv("Fake.csv");
data_true = pd.read_csv("True.csv");

# Processamento Iniziale Dati Fake


In [ ]:
# Prima
data_fake.head()

In [ ]:
# Eliminiamo le Colonne Subject e Date che non ci servono a nulla e aggiungiamo la colonna Label
data_fake.drop(["subject", "date"], axis=1, inplace=True)
data_fake["label"] = "FAKE"
data_fake.shape

In [ ]:
# Dopo
data_fake.head()

# Processamento Iniziale Dati True


In [ ]:
# Prima
data_true.head()

In [ ]:
# Eliminiamo le Colonne Subject e Date che non ci servono a nulla e aggiungiamo la colonna Label
data_true.drop(["subject", "date"], axis=1, inplace=True)
data_true["label"] = "TRUE"
data_true.shape

In [ ]:
# Dopo
data_true.head()

Nel caso dei Dati relativi alle Notizie Reali, dobbiamo effettuare un ulteriore processamento.
Come possiamo vedere dall'analisi dell'head(), in molte righe, la colonna Text contiene un riferimento iniziale alla fonte. Questo riferimento puo' inquinare i Dati (*Verrebbe trattato come un pattern, ma in realta' non lo e'*), quindi lo dobbiamo eliminare.
Il Modello deve prendere in considerazione solo il contenuto della Notizia.

# Concatenamento dei Dati e ulteriore Processamento

In [ ]:
# Concateniamo i DataSet data_fake e data_true in un nuovo dataset data_train
data_train = pd.concat([data_fake, data_true], axis=0);
data_train.shape

In [ ]:
# Prima - Inizio data_train
data_train.head()

In [ ]:
# Prima - Fine data_train
data_train.tail()

In [ ]:
def extract_txt(text):
  regex = re.search(r"(?<=\(Reuters\)\s\-\s).*", text)
  if regex: return regex.group(0)
  return text

# Rimuoviamo la Fonte dalle Righe che la presentano. Usiamo una Funzione:
data_train["text_processed"] = data_train["text"].apply(extract_txt)

# Cambiamo la Colonna Label: Fake = 1 | True = 0
data_train["label"] = data_train.label.apply(lambda x: 0 if x=="TRUE" else 1)

# Adesso dobbiamo solo unire la Colonna Titolo alla Colonna Testo
data_train["final_news"] = data_train["title"] + " " + data_train["text_processed"];

# Droppiammo le colonne che non ci interessano
data_train.drop(["text_processed", "text", "title"], axis=1, inplace=True);

In [ ]:
# Dopo - Inizio df
data_train.head()

In [ ]:
# Dopo - Fine df
data_train.tail()

# Processamento del Testo


In [ ]:
# Prima
data_train.head()

In [ ]:
# Dobbiamo liberarci delle stopwords presenti nel testo. Per farlo utilizziamo una libreria specifica
stop = stopwords.words("english");

def cleanText(text):
  # Lower Casing
  text = text.lower();

  # Removing Stopwords
  # (Mantieni solo le parole che non sono presenti in "stop" e
  # successivamente uniscile (join) mettendo uno spazio " " tra loro)
  text = " ".join([word for word in text.split() if word not in (stop)]);

  # Removing Non-Alphabetic Characters
  text = re.sub("[^a-z]", " ", text);

  return text;

data_train["final_news"] = data_train["final_news"].apply(cleanText)

In [ ]:
# Dopo
data_train.head()

# Configurazione del Modello LSTM
In questo caso di studio useremo Vettori Globali per la rappresentazione delle Parole (*Global Vectors for Word Representation "GLOVE"*).
GLOVE e' un tipo di apprendimento Non Supervisionato per ottenere la Rappresentazione Vettoriale delle Parole.

L'idea di base dietro l'Embedding e' che può essere utile a stabilire la rappresentazione contestuale delle Parole in un Documento, la loro semantica e le relazioni che condividono.

Funziona bene con LSTM in quanto il modello diviene più capace nel comprendere il testo in termini di conoscenza contestuale, rispetto ai metodi che sfruttano la sola vettorizzazione delle Parole.

In [ ]:
# File contenente l'embedding delle parole (GLOVE)
EMBEDDING_FILE = f"glove.6B.50d.txt"

# Configuration Settings
MAX_SEQUENCE_LENGTH = 100
MAX_VOCAB_SIZE = 20000
EMBEDDING_DIM = 50
VALIDATION_SPLIT = 0.2
BATCH_SIZE = 32
EPOCHS = 10

In [ ]:
y = data_train["label"].values;
X = data_train.drop(["label"], axis=1);
X.shape, y.shape

In [ ]:
# Load in Pre-Trained Word Vectors
print("Loading Word Vectors...")

word2vec = {}
with open(EMBEDDING_FILE) as f:
  # is just a space-separated text file in the format:
  # word: vec[0] vec[1] vec[2] ...
  # Ogni Parola ha associato un Vettore di Dimensione: EMBEDING_DIM=50
  for line in f:
    values = line.split()
    word = values[0]
    vec = np.asarray(values[1:], dtype="float32")
    word2vec[word] = vec
print("Found %s Word Vectors." % len(word2vec))


In [ ]:
# Convertiamo le Parole del nostro DataSet in Numeri Interi
# (Ad ogni parola viene associato un Numero Intero)
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE);
tokenizer.fit_on_texts(list(X["final_news"]))

# Adesso convertiamo tutte ogni riga di final_news in sequenze di Numeri
# Le Sequenze vengono create sfruttando ill tokenizer appena creato
X = tokenizer.texts_to_sequences(list(X["final_news"]))

# Limitiamo la lunghezza delle Sequenze per creare una Matrice di Dimenisioni N x T.
# Quando una Sequenza e' troppo corta ci mettiamo degli zeri di Padding
X = pad_sequences(X, maxlen=MAX_SEQUENCE_LENGTH);
print("Shape of Data Tensor:", X.shape)


In [ ]:
# Tokenizer: word -> integer mapping
word2idx = tokenizer.word_index
print("Found %s unique tokens." % len(word2idx))

In [ ]:
# Prepariamo la Matrice degli Embeddings
# In pratica prendiamo dal File di Embedding solo le Parole e i relativi Vettori
# che ci interessano
print("Inseriamo nella Matrice gli Embeddings Pre-Addestrati...");

num_words = min(MAX_VOCAB_SIZE, len(word2idx) + 1)
embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))

for word, i in word2idx.items():
  if i < MAX_VOCAB_SIZE:
    embedding_vector = word2vec.get(word)

    # Le parole non trovate nei file di Embedding saranno tutte 0
    if embedding_vector is not None:
      embedding_matrix[i] = embedding_vector

# **Modello di LSTM**



In [ ]:


# Carichiamo la Matrice degli Embeddings nell'Embedded Layer
# Settiamo trainable a False, in questo modo i valori di questo
# layer non cambieranno durante l'addestramento
embedding_layer = Embedding(
  num_words,
  EMBEDDING_DIM,
  weights=[embedding_matrix],
  input_length=MAX_SEQUENCE_LENGTH,
  trainable=False
);

# Costruiamo il Modello LSTM
input_ = Input(shape=(MAX_SEQUENCE_LENGTH,))
x = embedding_layer(input_)
# x = LSTM(15, return_sequences=True)(x)
x = Bidirectional(LSTM(15, return_sequences=True))(x)
x = GlobalMaxPool1D()(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(input_, output)
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# Dividiamo il DataSet in Train & Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0);

In [ ]:
# Addestriamo il Modello
print("Training Model...")

r = model.fit(
    X_train,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=VALIDATION_SPLIT
);

# Analisi dei Risultati

In [ ]:
# Plottiamo dei Grafici
acc = r.history['accuracy']
val_acc = r.history['val_accuracy']
loss = r.history['loss']
val_loss = r.history['val_loss']
epochs_range = range(1, len(r.epoch) + 1)

plt.figure(figsize=(15,5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Train Set')
plt.plot(epochs_range, val_acc, label='Val Set')
plt.legend(loc="best")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('LSTM Model Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Train Set')
plt.plot(epochs_range, val_loss, label='Val Set')
plt.legend(loc="best")
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('LSTM Model Loss')

plt.tight_layout()
plt.show()

In [ ]:
# Dimostriamo che non c'e' Overfitting
print("Accuracy of the model on Training Data is - " , model.evaluate(X_train,y_train)[1]*100)
print("Accuracy of the model on Testing Data is - " , model.evaluate(X_test,y_test)[1]*100)

In [ ]:
# Vediamo la Confusion Matrix
cm = confusion_matrix(y_test,pred.round())
cm = pd.DataFrame(cm , index = ['TRUE','FAKE'] , columns = ['TRUE','FAKE'])
plt.figure(figsize = (6,4))
sns.heatmap(cm,cmap= "Blues", linecolor = 'black' , linewidth = 1 , annot = True, fmt='' , xticklabels = ['TRUE','FAKE'] , yticklabels = ['TRUE','FAKE'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
print(classification_report(y_test, pred.round()))

# **Modello LSTM - Con Ottimizzazione degli Iperparametri**

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176,
    random_state=42,
    stratify=y_temp
)

# Verifica le dimensioni finali
print(f"Training set: {X_train.shape[0]} campioni")
print(f"Validation set: {X_val.shape[0]} campioni")
print(f"Test set: {X_test.shape[0]} campioni")


In [ ]:


# Configuration Settings di base
MAX_SEQUENCE_LENGTH = 100
EMBEDDING_DIM = 50

# 1. Definiamo la funzione builder per il modello
def build_model(hp):
    # Il livello di Embedding prende i tuoi dati pre-addestrati
    # Permettiamo al tuner di decidere se lasciarlo bloccato (False) o addestrarlo (True)
    trainable_embedding = hp.Boolean('trainable_embedding', default=False)

    embedding_layer = Embedding(
        num_words,
        EMBEDDING_DIM,
        weights=[embedding_matrix],
        input_length=MAX_SEQUENCE_LENGTH,
        trainable=trainable_embedding
    )

    input_ = Input(shape=(MAX_SEQUENCE_LENGTH,))
    x = embedding_layer(input_)

    # IPERPARAMETRO 1: Scegliamo il numero di unità LSTM tra 8 e 64
    lstm_units = hp.Int('lstm_units', min_value=8, max_value=64, step=8)
    x = Bidirectional(LSTM(lstm_units, return_sequences=True))(x)

    x = GlobalMaxPool1D()(x)
    output = Dense(1, activation="sigmoid")(x)

    model = Model(input_, output)

    # IPERPARAMETRO 2: Scegliamo il tipo di ottimizzatore
    optimizer_choice = hp.Choice('optimizer', values=['adam', 'rmsprop'])

    # IPERPARAMETRO 3: Scegliamo il Learning Rate dell'ottimizzatore
    lr = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')

    if optimizer_choice == 'adam':
        opt = Adam(learning_rate=lr)
    else:
        opt = RMSprop(learning_rate=lr)

    model.compile(
        loss="binary_crossentropy",
        optimizer=opt,
        metrics=["accuracy"]
    )

    return model

# 2. Configuriamo il Tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='hyperparam_tuning',
    project_name='fake_news_lstm'
)

# 3. Avviamo la ricerca
tuner.search(
    X_train, y_train,
    epochs=5,
    validation_data=(X_val, y_val),
    batch_size=32
)

# 4. Otteniamo il modello migliore e i parametri scelti
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"""
Il tuning è completato. Il numero ottimale di unità LSTM è {best_hps.get('lstm_units')},
l'ottimizzatore migliore è {best_hps.get('optimizer')} con un learning rate di {best_hps.get('learning_rate')}.
Embedding impostato su Trainable = {best_hps.get('trainable_embedding')}.
""")


In [ ]:
# Addestriamo il modello finale con i parametri migliori trovati
print("Training Model...")
model = tuner.hypermodel.build(best_hps)
r = model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val), batch_size=32)

# Analisi dei Risultati

In [ ]:
# Plottiamo dei Grafici
acc = r.history['accuracy']
val_acc = r.history['val_accuracy']
loss = r.history['loss']
val_loss = r.history['val_loss']
epochs_range = range(1, len(r.epoch) + 1)

plt.figure(figsize=(15,5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Train Set')
plt.plot(epochs_range, val_acc, label='Val Set')
plt.legend(loc="best")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('LSTM Model Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Train Set')
plt.plot(epochs_range, val_loss, label='Val Set')
plt.legend(loc="best")
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('LSTM Model Loss')

plt.tight_layout()
plt.show()

In [ ]:
# Dimostriamo che non c'e' Overfitting
print("Accuracy of the model on Training Data is - " , model.evaluate(X_train,y_train)[1]*100)
print("Accuracy of the model on Testing Data is - " , model.evaluate(X_test,y_test)[1]*100)
print("Accuracy of the model on Validation Data is - " , model.evaluate(X_val,y_val)[1]*100)

In [ ]:
# Vediamo la Confusion Matrix
cm = confusion_matrix(y_test,pred.round())
cm = pd.DataFrame(cm , index = ['TRUE','FAKE'] , columns = ['TRUE','FAKE'])
plt.figure(figsize = (6,4))
sns.heatmap(cm,cmap= "Blues", linecolor = 'black' , linewidth = 1 , annot = True, fmt='' , xticklabels = ['TRUE','FAKE'] , yticklabels = ['TRUE','FAKE'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
print(classification_report(y_test, pred.round()))